# NB02 — Airbnb Listings ETL

**Technologies:** Microsoft Fabric | PySpark | Delta Lake  
**Data Layer:** Raw CSV → Bronze → Silver

## 1. Objective

Clean and validate Airbnb listings data to support analysis of **listing supply, room types and indicative prices** across Melbourne's LGAs.

The Silver table also provides listing attributes and snapshot dates for downstream ETL processing.

## 2. Data Sources & Outputs

| Type | Source |
|---|---|
| Raw Input | `Files/listings 2.csv` |
| Bronze Output | `dbo.bronze_listings` |
| Silver Output | `dbo.silver_listings` |
| Downstream Gold | `dbo.dim_listing` |

**Data Grain:** One record per unique `listing_id` in the source snapshot.

**Load Strategy:** Full refresh (`overwrite`).

## 3. ETL Process

1. **Data Ingestion:** Read the CSV using PySpark with multiline parsing settings.
2. **Bronze Storage:** Save ingested data to `dbo.bronze_listings` in Delta format.
3. **Data Cleaning:** Standardise data types, listing IDs, LGA names, room types and prices.
4. **Data Transformation:** Prepare listing attributes, snapshot dates, price quotes and validity indicators.
5. **Data Quality Validation:** Check record counts, invalid IDs, duplicate listings and required fields.
6. **Silver Storage:** Write validated data to `dbo.silver_listings` in Delta format.

## 4. CSV Parsing & Data Quality

**Issue:** Embedded line breaks in CSV text fields caused incorrect parsing.

| Metric | Incorrect Parsing | Corrected Parsing |
|---|---:|---:|
| Records | 46,525 | 25,715 |
| Invalid Listing IDs | 20,810 | 0 |
| Duplicate Listing IDs | — | 0 |

**Design Decision:** Use corrected multiline CSV parsing and validation checks. Critical failures stop the ETL activity.

The debugging investigation is documented separately in `docs/csv_parsing_debugging.md`.

## 5. Business Definitions

| Field | Meaning |
|---|---|
| `listing_id` | Unique Airbnb listing identifier |
| `lga` | Local Government Area |
| `room_type` | Accommodation category |
| `quote_price_aud` | Indicative price for a captured booking period |
| `has_valid_quote` | Indicates whether a valid quote is available |
| `snapshot_date` | Date associated with the listing snapshot |

**Pricing Limitation:** Captured quotes are not available for every listing. They represent a particular booking period, not a daily price series or confirmed transaction price.

## 6. Validated Output

| Metric | Result |
|---|---:|
| Unique Listings | 25,715 |
| Invalid Listing IDs | 0 |
| Duplicate Listing IDs | 0 |
| Valid Price Quotes | 19,004 |
| Price Quote Coverage | 73.90% |

The cleaned data supports downstream listing-level analysis and provides the snapshot date used by Calendar ETL.

## 7. Pipeline Dependency & Updates

**NB02 Listings → NB01 Calendar → NB04 Gold**

NB01 uses the latest Listings snapshot date to define its 90-day analysis window. NB04 uses the validated Listings data to build `dbo.dim_listing`.

**Full Refresh:** New snapshots must use the expected OneLake file path and a compatible schema. The pipeline must be triggered after uploading, and previous table contents are overwritten rather than retained as historical snapshots.

In [ ]:
# Import Required Libraries
from pyspark.sql import functions as F

In [ ]:
# Set the source path for the Listings CSV file
source_path = "Files/listings 2.csv"

# Correct CSV parser. Never run the intentionally broken parser here.
df_fixed = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .load(source_path)
)

# Validate source columns needed by all downstream transformations
required_columns = [
    "id", "name", "host_id", "neighbourhood_cleansed",
    "room_type", "property_type", "accommodates", "latitude", "longitude",
    "price", "price_quote_price_per_night", "price_quote_checkin_date",
    "price_quote_checkout_date", "last_scraped",
    "review_scores_rating"
]
missing_columns = sorted(set(required_columns) - set(df_fixed.columns))
assert not missing_columns, f"Required source columns missing: {missing_columns}"

bronze_rows = df_fixed.count()
invalid_ids = df_fixed.filter(
    F.col("id").isNull() |
    (~F.trim(F.col("id")).rlike(r"^[0-9]+$"))
).count()

duplicate_id_groups = (
    df_fixed.groupBy("id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Source rows:", bronze_rows)
print("Source columns:", len(df_fixed.columns))
print("Invalid listing IDs:", invalid_ids)
print("Duplicate listing ID groups:", duplicate_id_groups)

assert bronze_rows > 0, "Empty Listings CSV!"
assert invalid_ids == 0, "Malformed or missing listing IDs; check CSV parsing!"
assert duplicate_id_groups == 0, "Duplicate listing IDs in this snapshot!"

print("Listings CSV parsing and source validation passed!")

StatementMeta(, add77c5d-5cae-4e88-a54f-20f8d7e159d8, 3, Finished, Available, Finished, False)

Source rows: 25715
Source columns: 90
Invalid listing IDs: 0
Duplicate listing ID groups: 0
Listings CSV parsing and source validation passed!


## 2. Write the validated Bronze table

Overwrite Bronze only after CSV parsing and source-level checks have succeeded. The Bronze table retains the raw imported fields; Silver below handles business-focused typing and cleaning.

In [ ]:
# Write the cleaned DataFrame to a Delta table in the bronze layer
(
    df_fixed.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dbo.bronze_listings")
)

raw = spark.table("dbo.bronze_listings")
assert raw.count() == bronze_rows, "Bronze row count changed after writing!"
print("Bronze Listings saved successfully!")

StatementMeta(, add77c5d-5cae-4e88-a54f-20f8d7e159d8, 4, Finished, Available, Finished, False)

Bronze Listings saved successfully!


## 3. Missing value assessment

Missing prices are **not** replaced with zero, and listings are **not** dropped simply because their prices are unavailable. Price coverage is reported when comparing pricing across LGAs.

In [ ]:
# Check for missing values in critical columns
check_cols = [
    "price", "price_quote_price_per_night",
    "review_scores_rating", "property_type"
]

missing_stats = raw.select(*[
    F.sum(
        F.when(
            F.col(c).isNull() |
            (F.trim(F.col(c).cast("string")) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in check_cols
]).first()

for c in check_cols:
    missing_count = missing_stats[c]
    print(f"{c}: {missing_count:,} missing ({missing_count / bronze_rows * 100:.2f}%)")

StatementMeta(, add77c5d-5cae-4e88-a54f-20f8d7e159d8, 5, Finished, Available, Finished, False)

price: 6,711 missing (26.10%)
price_quote_price_per_night: 6,711 missing (26.10%)
review_scores_rating: 4,585 missing (17.83%)
property_type: 0 missing (0.00%)


## 4. Data transformation

Converts listing/host IDs, numbers and dates; normalises the LGA name; and creates cleaned price columns and `has_valid_quote`. The quote is one indicative listing-level price, **not** a daily Calendar price.

In [4]:
# Convert price strings (e.g. "$150.00") into numbers.
# Keep missing or invalid prices as NULL.
def clean_price(column_name):
    value = F.trim(F.col(column_name))

    numeric_price = F.regexp_replace(
        value, r"[$,]", ""
    ).cast("double")

    return F.when(
        value.rlike(r"^\$?[0-9][0-9,]*(\.[0-9]+)?$") &
        (numeric_price > 0),
        numeric_price
    ).otherwise(F.lit(None).cast("double"))

# Select and clean relevant columns
silver_listings = raw.select(
    # Listing identification
    F.col("id").cast("long").alias("listing_id"),
    F.trim(F.col("name")).alias("listing_name"),
    F.col("host_id").cast("long").alias("host_id"),

    # Location and property type
    F.trim(F.col("neighbourhood_cleansed")).alias("lga"),
    F.trim(F.col("room_type")).alias("room_type"),
    F.trim(F.col("property_type")).alias("property_type"),
    F.col("accommodates").cast("int").alias("accommodates"),

    # Geographic coordinates
    F.col("latitude").cast("double").alias("latitude"),
    F.col("longitude").cast("double").alias("longitude"),

    # Pricing
    clean_price("price").alias("listing_price_aud"),
    clean_price("price_quote_price_per_night").alias(
        "quote_price_aud"
    ),

    # Quote and snapshot dates
    F.to_date("price_quote_checkin_date").alias(
        "quote_checkin_date"
    ),
    F.to_date("price_quote_checkout_date").alias(
        "quote_checkout_date"
    ),
    F.to_date("last_scraped").alias("snapshot_date")
)

# Convert blank text values to NULL
for c in ["listing_name", "lga", "room_type", "property_type"]:
    silver_listings = silver_listings.withColumn(
        c,
        F.when(
            F.length(F.trim(F.col(c))) > 0,
            F.col(c)
        ).otherwise(F.lit(None))
    )

# Add price availability flag
silver_listings = silver_listings.withColumn(
    "has_valid_quote",
    F.when(F.col("quote_price_aud").isNotNull(), 1)
     .otherwise(0)
)

# Preview cleaned data
silver_listings.printSchema()
display(silver_listings.limit(10))

StatementMeta(, add77c5d-5cae-4e88-a54f-20f8d7e159d8, 6, Finished, Available, Finished, False)

root
 |-- listing_id: long (nullable = true)
 |-- listing_name: string (nullable = true)
 |-- host_id: long (nullable = true)
 |-- lga: string (nullable = true)
 |-- room_type: string (nullable = true)
 |-- property_type: string (nullable = true)
 |-- accommodates: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- listing_price_aud: double (nullable = true)
 |-- quote_price_aud: double (nullable = true)
 |-- quote_checkin_date: date (nullable = true)
 |-- quote_checkout_date: date (nullable = true)
 |-- snapshot_date: date (nullable = true)
 |-- has_valid_quote: integer (nullable = false)



SynapseWidget(Synapse.DataFrame, 4f980ee0-42db-447b-9dbd-aead11b9dad4)

## 5. Dynamic data quality checks and Silver write

Enforce **one row per valid listing** without hard-coding the snapshot count. All listings must have an LGA, room type and valid snapshot date. Prices may be null. The Notebook raises an exception on failure so the Data Factory Pipeline's **On success** dependency cannot continue to Gold with stale or invalid Silver data.

In [5]:
total_listings = silver_listings.count()
unique_ids = silver_listings.select("listing_id").distinct().count()
missing_ids = silver_listings.filter(F.col("listing_id").isNull()).count()
missing_lga = silver_listings.filter(F.col("lga").isNull()).count()
missing_room_type = silver_listings.filter(F.col("room_type").isNull()).count()
missing_snapshot_dates = silver_listings.filter(F.col("snapshot_date").isNull()).count()
valid_quotes = silver_listings.filter(F.col("has_valid_quote") == 1).count()

print(f"Bronze Rows: {bronze_rows:,}")
print(f"Silver Rows: {total_listings:,}")
print(f"Unique Listing IDs: {unique_ids:,}")
print(f"Missing IDs: {missing_ids:,}")
print(f"Missing LGAs: {missing_lga:,}")
print(f"Missing Room Types: {missing_room_type:,}")
print(f"Missing Snapshot Dates: {missing_snapshot_dates:,}")
print(f"Valid Quoted Prices: {valid_quotes:,}")
print(f"Quote Coverage: {valid_quotes / total_listings * 100:.2f}%")

assert total_listings > 0, "Silver Listings table is empty!"
assert total_listings == bronze_rows, "Bronze-to-Silver listing count mismatch!"
assert unique_ids == total_listings, "Duplicate listing IDs in Silver!"
assert missing_ids == 0, "Missing listing IDs in Silver!"
assert missing_lga == 0, "Missing LGA in Silver!"
assert missing_room_type == 0, "Missing room types in Silver!"
assert missing_snapshot_dates == 0, "Missing or invalid snapshot dates in Silver!"

(
    silver_listings.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dbo.silver_listings")
)

print("Silver Listings validated and saved successfully!")
display(silver_listings.limit(10))

StatementMeta(, add77c5d-5cae-4e88-a54f-20f8d7e159d8, 7, Finished, Available, Finished, False)

Bronze Rows: 25,715
Silver Rows: 25,715
Unique Listing IDs: 25,715
Missing IDs: 0
Missing LGAs: 0
Missing Room Types: 0
Missing Snapshot Dates: 0
Valid Quoted Prices: 19,004
Quote Coverage: 73.90%
Silver Listings validated and saved successfully!


SynapseWidget(Synapse.DataFrame, a7ed42e1-00b6-46eb-bc5e-4cbed80e70c3)

## Outcome and limitations

- Data Factory should run this Notebook **before** the Calendar ETL, which reads `dbo.silver_listings.snapshot_date`.
- Bronze and Silver are full-refresh representations of **the supplied snapshot**. This Notebook does not download a newer source file and does not preserve historical snapshots.
- Missing price quotes remain null. Report quote coverage when calculating indicative nightly prices.
- Calendar availability describes **advertised availability**, not observed bookings or occupancy.
- Historical Sep 2026 results (25,715 source rows, 46,525 rows from the defective parser, and 73.90% quote coverage) are documented in the GitHub CSV parsing debugging case study, not used as hard-coded validation thresholds.